# Description Text Experiment: Does the Listing Description Help?

Standalone experiment -- does **not** touch `main.py` / `notebooks/pipeline.ipynb`.

Builds on `notebooks/sentiment_features_experiment.ipynb` (baseline 65.15% ->
+ aspect sentiment 66.00% R^2 for XGBoost). This notebook adds one more
variant: TF-IDF features extracted from the listing `description` text itself
(currently only `desc_word_count` -- a word count -- is used, never the actual
content). The vectorizer is fit on the training split's descriptions only, to
avoid leaking test-set vocabulary/IDF weights into training.

- **A: baseline** (numeric + encoded categoricals)
- **B: + aspect sentiment** (host quality / cleanliness / location / amenities + review count)
- **C: + aspect sentiment + description TF-IDF** (top 30 words, fit on train only)

All three variants share the exact same train/test split.


## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import CLEANED_TARGET_PATH, RANDOM_STATE, RAW_LISTINGS_PATH, TARGET_COLUMN, TEST_SIZE
from src.data.clean import clean_missing_target
from src.data.load import load_reviews
from src.evaluate import print_metrics_log_target
from src.features.build_features import build_feature_matrix
from src.features.categorical import fit_categorical_encoders, transform_categorical_features
from src.features.sentiment import compute_aspect_sentiment
from src.features.text import fit_tfidf_vectorizer, transform_tfidf_features
from src.models.tree_models import train_xgboost


## Step 1: Build the same feature set as the main pipeline (numeric + categorical)

In [2]:
clean_missing_target(RAW_LISTINGS_PATH, CLEANED_TARGET_PATH, target_column=TARGET_COLUMN)
df = pd.read_csv(CLEANED_TARGET_PATH)
df = build_feature_matrix(df)
df.shape


Reading /Users/jnanadeep/Desktop/airbnb-price-prediction/data/raw/listings.csv...



--- Processing Summary ---
Initial row count:          40769
Rows with missing 'price': 953 (Removed)
Remaining row count:        39816
Cleaned data saved to:      /Users/jnanadeep/Desktop/airbnb-price-prediction/data/processed/airbnb_rio_cleaned_target.csv



Dropping columns with >90.0% missing values:
 - neighborhood_overview
 - host_since
 - host_response_time
 - host_response_rate
 - host_acceptance_rate
 - host_thumbnail_url
 - host_neighbourhood
 - host_total_listings_count
 - host_verifications
 - neighbourhood
 - neighbourhood_group_cleansed
 - calendar_updated
 - license
 - instant_bookable

Imputed 25 numerical columns using their median value.
--- Running Feature Selection ---
Original unique columns: 76
Filtered columns retained: 24
--- Running Dynamic Outlier Removal ---
Calculated 99.0th percentile threshold: $6,999.13
Removed 399 listing(s) outside the range [$10 - $6,999.13].
New maximum price in dataset: $6,990.00


(39417, 27)

## Step 2: Aspect-based review sentiment (full reviews file, as validated in the sentiment experiment)

In [3]:
reviews = load_reviews()
aspect_sentiment = compute_aspect_sentiment(reviews)
aspect_score_cols = [c for c in aspect_sentiment.columns if c != "listing_id"]

df_sentiment = df.merge(aspect_sentiment, left_on="id", right_on="listing_id", how="left").drop(columns=["listing_id"])
df_sentiment[aspect_score_cols] = df_sentiment[aspect_score_cols].fillna(0)
df_sentiment.shape


(39417, 32)

## Step 3: Shared train/test split (same rows for all three variants)

In [4]:
y_log = np.log1p(df_sentiment[TARGET_COLUMN])
train_df, test_df, y_train_log, y_test_log = train_test_split(
    df_sentiment, y_log, test_size=TEST_SIZE, random_state=RANDOM_STATE
)


## Step 4: Build the three feature matrices

In [5]:
numeric_cols = df.drop(columns=[TARGET_COLUMN, "id"], errors="ignore").select_dtypes(include=[np.number]).columns

encoders = fit_categorical_encoders(train_df)
train_cat = transform_categorical_features(train_df, encoders)
test_cat = transform_categorical_features(test_df, encoders)

# Variant A: baseline (numeric + categorical)
X_train_a = pd.concat([train_df[numeric_cols], train_cat], axis=1)
X_test_a = pd.concat([test_df[numeric_cols], test_cat], axis=1)

# Variant B: + aspect sentiment
X_train_b = pd.concat([X_train_a, train_df[aspect_score_cols]], axis=1)
X_test_b = pd.concat([X_test_a, test_df[aspect_score_cols]], axis=1)

# Variant C: + aspect sentiment + description TF-IDF (fit on train only)
tfidf_vectorizer = fit_tfidf_vectorizer(train_df, column="description", max_features=30)
train_tfidf = transform_tfidf_features(train_df, tfidf_vectorizer, column="description")
test_tfidf = transform_tfidf_features(test_df, tfidf_vectorizer, column="description")

X_train_c = pd.concat([X_train_b, train_tfidf], axis=1)
X_test_c = pd.concat([X_test_b, test_tfidf], axis=1)

variants = {
    "A: baseline": (X_train_a, X_test_a),
    "B: + aspect sentiment": (X_train_b, X_test_b),
    "C: + aspect sentiment + description TF-IDF": (X_train_c, X_test_c),
}


## Step 5: Train XGBoost on each variant

In [6]:
results = {}
models = {}
for name, (X_train, X_test) in variants.items():
    model = train_xgboost(X_train, y_train_log)
    preds_log = model.predict(X_test)
    metrics = print_metrics_log_target(name, y_test_log, preds_log)
    metrics["n_features"] = X_train.shape[1]
    results[name] = metrics
    models[name] = (model, X_train.columns)


A: baseline                    | R^2 Score: 65.15% | MAE: $248.73 | RMSE: $502.66


B: + aspect sentiment          | R^2 Score: 66.00% | MAE: $245.14 | RMSE: $498.81


C: + aspect sentiment + description TF-IDF | R^2 Score: 65.25% | MAE: $247.23 | RMSE: $502.13


## Step 6: Summary

In [7]:
comparison = pd.DataFrame(results).T[["n_features", "mae", "rmse", "r2"]]
comparison


,n_features,mae,rmse,r2
A: baseline,38.0,248.729040,502.662966,0.651514
B: + aspect sentiment,43.0,245.138759,498.810945,0.659976
C: + aspect sentiment + description TF-IDF,73.0,247.226752,502.127027,0.652485


## Step 7: Do any description TF-IDF words crack the top features?

In [8]:
model, cols = models["C: + aspect sentiment + description TF-IDF"]
importances = pd.Series(model.feature_importances_, index=cols)
importances.sort_values(ascending=False).head(20)


bedrooms                            0.340092
room_type_Entire home/apt           0.080067
bathrooms                           0.077567
room_type_Shared room               0.048958
total_reviews_count                 0.043656
room_type_Private room              0.037918
accommodates                        0.023566
dist_to_beach                       0.023263
reviews_per_month                   0.016339
latitude                            0.016296
location_safety_score               0.014899
number_of_reviews                   0.014024
property_type_Room in hotel         0.013861
neighbourhood_cleansed_frequency    0.012619
amenities_score                     0.009432
availability_365                    0.009404
review_scores_rating                0.009190
longitude                           0.008608
hosts_time_as_host_years            0.007988
host_listings_count                 0.006353
dtype: float32